In [ ]:
import pandas as pd
import numpy as np
import nltk
import torch
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


device(type='cuda')

In [5]:
# Load dataset
dataset = pd.read_csv("/kaggle/input/news-headline/news_summary.csv", encoding='latin-1')

In [6]:
dataset.columns

Index(['headlines', 'text'], dtype='object')

In [7]:
# Concatenate topic label with text
dataset["input_text"] = "Given the following news article, write an informative headline: " + "\n\nText: " + dataset["text"]

# Display Results
print(dataset[['input_text']].head(10))

                                          input_text
0  Given the following news article, write an inf...
1  Given the following news article, write an inf...
2  Given the following news article, write an inf...
3  Given the following news article, write an inf...
4  Given the following news article, write an inf...
5  Given the following news article, write an inf...
6  Given the following news article, write an inf...
7  Given the following news article, write an inf...
8  Given the following news article, write an inf...
9  Given the following news article, write an inf...


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import pandas as pd

# Load a tokenizer
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.float16)

# Tokenize and get token lengths
dataset["text_token_length"] = dataset["text"].astype(str).apply(lambda x: len(tokenizer.tokenize(x)))
dataset["headline_token_length"] = dataset["headlines"].astype(str).apply(lambda x: len(tokenizer.tokenize(x)))

# Get min and max token lengths
min_text_length = dataset["text_token_length"].min()
max_text_length = dataset["text_token_length"].max()
min_headline_length = dataset["headline_token_length"].min()
max_headline_length = dataset["headline_token_length"].max()

# Print the results
print(f"Text Length - Min: {min_text_length}, Max: {max_text_length}")
print(f"Headline Length - Min: {min_headline_length}, Max: {max_headline_length}")


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Text Length - Min: 54, Max: 187
Headline Length - Min: 5, Max: 41


In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorForSeq2Seq
from sklearn.model_selection import train_test_split

class NewsHeadlineDataset(Dataset):
    def __init__(self, input_texts, headlines, tokenizer, article_max_length=200, headline_max_length=50):
        self.input_texts = input_texts
        self.headlines = headlines
        self.tokenizer = tokenizer
        self.article_max_length = article_max_length
        self.headline_max_length = headline_max_length

    def __len__(self):
        return len(self.input_texts)

    def __getitem__(self, idx):
        input_text = self.input_texts[idx]
        headline = self.headlines[idx]

        # Tokenize input and target without returning tensors
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.article_max_length,
            padding='max_length',
            truncation=True,
        )
        target_encoding = self.tokenizer(
            headline,
            max_length=self.headline_max_length,
            padding='max_length',
            truncation=True,
        )

        return {
            'input_ids': input_encoding['input_ids'],
            'attention_mask': input_encoding['attention_mask'],
            'labels': target_encoding['input_ids']
        }

# Assuming df is your DataFrame with columns: 'headlines', 'input_text'
headlines = dataset['headlines'].tolist()
input_texts = dataset['input_text'].tolist()

# Split dataset into train, validation, and test sets
train_texts, test_texts, train_headlines, test_headlines = train_test_split(
    input_texts, headlines, test_size=0.15, shuffle=True, random_state=12
)
train_texts, val_texts, train_headlines, val_headlines = train_test_split(
    train_texts, train_headlines, test_size=0.10, shuffle=True, random_state=12
)

def get_dataloader(input_texts, headlines, tokenizer, batch_size=6, shuffle=False):
    dataset = NewsHeadlineDataset(input_texts, headlines, tokenizer)
    data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        collate_fn=data_collator,
        pin_memory=True,
        shuffle=shuffle
    )

# Create DataLoaders
train_dataloader = get_dataloader(train_texts, train_headlines, tokenizer, shuffle=True)
val_dataloader = get_dataloader(val_texts, val_headlines, tokenizer)
test_dataloader = get_dataloader(test_texts, test_headlines, tokenizer)

print(f"Train set size: {len(train_texts)}")
print(f"Validation set size: {len(val_texts)}")
print(f"Test set size: {len(test_texts)}")

Train set size: 75276
Validation set size: 8364
Test set size: 14760


In [ ]:
from nltk.translate.meteor_score import meteor_score
import torch
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def evaluate_meteor(model, tokenizer, test_texts, test_headlines):
    meteor_scores = []

    for text, reference in tqdm(zip(test_texts, test_headlines), total=len(test_texts)):
        # Tokenize input text and move to device
        input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)

        # Generate headline using the fine-tuned model
        with torch.no_grad():
            output_ids = model.generate(input_ids=input_ids, max_length=50)

        generated_headline = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        # Compute METEOR score
        score = meteor_score([reference], generated_headline)
        meteor_scores.append(score)

    # Calculate average METEOR score
    avg_meteor = sum(meteor_scores) / len(meteor_scores)
    return avg_meteor

# Evaluate the model using METEOR metric
avg_meteor_score = evaluate_meteor(model, tokenizer, test_texts, test_headlines)

print(f"Average METEOR Score: {avg_meteor_score:.4f}")


100%|██████████| 14760/14760 [2:35:26<00:00,  1.58it/s]  

Average METEOR Score: 0.3262


In [ ]:
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from transformers import T5ForConditionalGeneration

# LoRA config
lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    lora_dropout=0.1,
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=["q", "v"],
    bias="none"
)

# Load the base model (T5)
base_model = T5ForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.float16)

# Apply LoRa Config
model = get_peft_model(base_model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

# Disable caching to avoid memory issues
model.config.use_cache = False

# Optional: Enable gradient checkpointing for memory efficiency
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# Move to GPU
model = model.to(device)

trainable params: 3,538,944 || all params: 226,442,496 || trainable%: 1.5628


In [19]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback

# Create training and validation datasets using the custom dataset class
train_dataset = NewsHeadlineDataset(
    input_texts=train_texts,
    headlines=train_headlines,
    tokenizer=tokenizer,
    article_max_length=200,
    headline_max_length=50
)

val_dataset = NewsHeadlineDataset(
    input_texts=val_texts,
    headlines=val_headlines,
    tokenizer=tokenizer,
    article_max_length=200,
    headline_max_length=50
)

test_dataset = NewsHeadlineDataset(
    input_texts=test_texts,
    headlines=test_headlines,
    tokenizer=tokenizer,
    article_max_length=200,
    headline_max_length=50
)

# Data collator for dynamic padding
data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True)

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-lora-finetuning",
    num_train_epochs=10,
    learning_rate=1e-4,
    warmup_steps=500,
    auto_find_batch_size=True,
    logging_steps=500,
    eval_strategy="epoch",
    save_strategy="epoch",
    weight_decay=0.01,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
)


# Initialize the Seq2SeqTrainer with model, training args, datasets, tokenizer, and data collator
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.461600,0.402733
2,0.425500,0.384994
3,0.422800,0.375963
4,0.410300,0.369548
5,0.397500,0.368346
6,0.392300,0.363893
7,0.391100,0.361413
8,0.385900,0.359481
9,0.385600,0.360234
10,0.378200,0.359560


TrainOutput(global_step=94100, training_loss=0.4411161486780732, metrics={'train_runtime': 38031.1635, 'train_samples_per_second': 19.793, 'train_steps_per_second': 2.474, 'total_flos': 1.82258935308288e+17, 'train_loss': 0.4411161486780732, 'epoch': 10.0})

In [ ]:
from transformers import T5ForConditionalGeneration
from peft import PeftModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Your saved checkpoint directory
checkpoint_path = "/kaggle/input/t5-base-lora-8-10/pytorch/default/1"

# Load base model
base_model = T5ForConditionalGeneration.from_pretrained("t5-base", torch_dtype=torch.float16)

# Load LoRA adapter weights
model = PeftModel.from_pretrained(base_model, checkpoint_path).to(device)

# Move model to appropriate device
model = model.to(device)
model.eval()  # Set to evaluation mode


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32128, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32128, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=32, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=32, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
             

In [ ]:
from nltk.translate.meteor_score import meteor_score
import torch
from tqdm import tqdm

def evaluate_meteor(model, tokenizer, test_texts, test_headlines):
    meteor_scores = []

    for text, reference in tqdm(zip(test_texts, test_headlines), total=len(test_texts)):
        # Tokenize input text and move to device
        input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)

        # Generate headline using the fine-tuned model
        with torch.no_grad():
            output_ids = model.generate(input_ids=input_ids, max_length=50)

        generated_headline = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        # Compute METEOR score
        score = meteor_score([reference], generated_headline)
        meteor_scores.append(score)

    # Calculate average METEOR score
    avg_meteor = sum(meteor_scores) / len(meteor_scores)
    return avg_meteor

# Evaluate the model using METEOR metric
avg_meteor_score = evaluate_meteor(model, tokenizer, test_texts, test_headlines)

print(f"Average METEOR Score: {avg_meteor_score:.4f}")

100%|██████████| 14760/14760 [1:44:24<00:00,  2.36it/s]

Average METEOR Score: 0.4778
